In [23]:
import torch 
import numpy as np 
import matplotlib.pyplot as plt 
import sys
import os
import joblib

sys.path.insert(0, os.path.abspath("../../.."))
sys.path.insert(0, os.path.abspath("../.."))

from sklearn.decomposition import PCA 
from cartpole_wind import cartpole_model_factory, make_envs, sample_tasks_eval
from utils.evaluation import run_meta_eval 


# Load the K Means model
kmeans = joblib.load("../../../checkpoints/cartpole_wind_kmeans.joblib")

# Load the original cluster assignments
cluster_labels = np.load("../../../checkpoints/cartpole_wind_cluster_labels.npy")



In [24]:
# Create new environments using the same 20 wind values
repeat_env_configs = sample_tasks_eval(num_envs=20)
repeat_envs = make_envs(repeat_env_configs)

In [25]:
# Initialize the trained RL2 model
model = cartpole_model_factory()

checkpoint_path = os.path.join("..", "..", "..", "checkpoints", "cartpole_wind.pt")
checkpoint = torch.load(checkpoint_path, map_location=torch.device("cpu"))

if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)


In [26]:
# Define the number of environments
repeat_num_envs = len(repeat_env_configs)
steps_per_env = 3000
model.eval()
repeat_obs, _ = repeat_envs.reset()

repeat_hidden_state = torch.zeros(
    1, repeat_num_envs, model.feat_extractor.h_dim
)

repeat_prev_action = torch.zeros(repeat_num_envs, dtype=torch.long)
repeat_prev_reward = torch.zeros(repeat_num_envs, 1)
repeat_prev_done = torch.ones(repeat_num_envs, 1)

# Store the hidden states separately for each environment
repeat_env_hidden_accumulator = [[] for _ in range(repeat_num_envs)]

# Collect hidden states during the rollout
with torch.no_grad():
    for step in range(steps_per_env):
        logits, values, repeat_hidden_state = model(
            torch.tensor(repeat_obs).view(1, repeat_num_envs, -1).float(),
            repeat_prev_action.view(1, repeat_num_envs),
            repeat_prev_reward.view(1, repeat_num_envs, -1).float(),
            repeat_prev_done.view(1, repeat_num_envs, 1),
            repeat_hidden_state
        )

        current_h = repeat_hidden_state.squeeze(0).cpu().numpy()

        for i in range(repeat_num_envs):
            repeat_env_hidden_accumulator[i].append(current_h[i])

        actions = torch.argmax(logits, dim=-1)
        actions_env = actions.squeeze(0).cpu().numpy()

        repeat_obs, reward, terminated, truncated, info = repeat_envs.step(
            actions_env
        )

        repeat_prev_action = actions
        repeat_prev_reward = torch.tensor(reward).unsqueeze(1).float()

        # Update done flags for the next step
        done = np.logical_or(terminated, truncated)
        repeat_prev_done = torch.tensor(done, dtype=torch.float32).unsqueeze(1)

In [27]:
# Calculate the average hidden state for each repeated environment
repeat_avg_hidden_states = np.array([
    np.mean(h, axis=0) for h in repeat_env_hidden_accumulator
])

print(repeat_avg_hidden_states.shape)

(20, 64)


In [28]:
# Assign the repeated hidden states to the existing K-Means clusters
repeat_cluster_labels = kmeans.predict(repeat_avg_hidden_states)

# Compare the original and repeated cluster assignments
for i, cfg in enumerate(repeat_env_configs):
    print(
        f"Wind = {cfg['wind_mag']:.2f} | "
        f"Original Cluster = {cluster_labels[i]} | "
        f"Repeated Cluster = {repeat_cluster_labels[i]}"
    )

# Calculate the percentage of matching cluster assignments
agreement = np.mean(cluster_labels == repeat_cluster_labels)

print(f"\nCluster Agreement: {agreement:.2%}")

Wind = -5.00 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -4.47 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -3.95 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -3.42 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -2.89 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -2.37 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -1.84 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -1.32 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -0.79 | Original Cluster = 1 | Repeated Cluster = 1
Wind = -0.26 | Original Cluster = 2 | Repeated Cluster = 2
Wind = 0.26 | Original Cluster = 1 | Repeated Cluster = 1
Wind = 0.79 | Original Cluster = 1 | Repeated Cluster = 1
Wind = 1.32 | Original Cluster = 2 | Repeated Cluster = 2
Wind = 1.84 | Original Cluster = 2 | Repeated Cluster = 2
Wind = 2.37 | Original Cluster = 2 | Repeated Cluster = 2
Wind = 2.89 | Original Cluster = 2 | Repeated Cluster = 2
Wind = 3.42 | Original Cluster = 2 | Repeated Cluster = 2
Wind